# Task 3: Document Best-Performing Models for Deployment
**SalaryInsight: Predictive Modeling for Informed Salary Predictions**

## Summary of findings
**Problem:** predict an employee's salary band (Low / Medium / High, tertiles of `Salary_LPA`) from 11 features. 2,000 employees; 25 missing values each in `Performance_Rating` and `Training_Hours` (median-imputed). 80/20 stratified split, `random_state=42`.

| Model | Default CV Acc (Task 1) | Tuned CV F1 (Task 2) | Tuned Test Acc |
|---|---|---|---|
| **SVM (RBF)** | 0.851 | **0.869** | 0.873 |
| Logistic Regression | 0.859 | 0.866 | 0.883 |
| Random Forest | 0.808 | 0.813 | 0.835 |
| KNN | 0.743 | - | - |
| Decision Tree | 0.729 | - | - |

**Misclassification trends**
- Almost all errors are between *adjacent* bands; the tuned SVM makes no Low↔High errors on the test set.
- **Medium** is the hardest band (SVM recall 0.86, precision 0.78).
- Decision Tree/KNN make ~60% more errors than SVM/Logistic Regression: salary depends on smooth, near-linear effects.

## Recommendation
1. **Deploy the tuned SVM (RBF, C=10, gamma=0.01)**: best cross-validated macro-F1 (0.869), ~87% test accuracy.
2. **Interpretable alternative: tuned Logistic Regression.** It is statistically tied with SVM (test accuracy 0.883). Use it if HR/audit needs to explain each prediction.
3. Do not deploy Random Forest, KNN or Decision Tree (clearly less accurate).
4. **Caveats:** predictions are salary *bands*, not exact salaries; only 2,000 rows. Monitor accuracy on new data and retrain periodically.

## Reproducible code
Requirements: `python>=3.9, pandas, scikit-learn, joblib, matplotlib`. Keep `employee_data.csv` in the same folder.

In [1]:
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

RS = 42
LABELS = ["Low", "Medium", "High"]
import joblib
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

### Rebuild and verify the recommended models (best parameters from Task 2)

In [2]:
df = pd.read_csv("employee_data.csv")
y = pd.qcut(df["Salary_LPA"], 3, labels=LABELS)
X = df.drop(columns=["Salary_LPA"])
num = X.select_dtypes("number").columns.tolist()
cat = [c for c in X.columns if c not in num]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)

final = {
  "SVM (recommended)":            SVC(kernel="rbf", C=10, gamma=0.01, random_state=RS),
  "Logistic Regression (alt.)":   LogisticRegression(C=100, max_iter=3000),
}
rows, fitted = [], {}
for n, clf in final.items():
    m = Pipeline([("pre", pre), ("clf", clf)]).fit(Xtr, ytr); fitted[n] = m
    p = m.predict(Xte)
    rows.append({"Model": n, "Test Acc": round(accuracy_score(yte, p), 4),
                 "Test F1": round(f1_score(yte, p, average="macro"), 4)})
pd.DataFrame(rows)

,Model,Test Acc,Test F1
0,SVM (recommended),0.8725,0.8736
1,Logistic Regression (alt.),0.8825,0.8827


In [3]:
# Save the recommended model for deployment
joblib.dump(fitted["SVM (recommended)"], "best_salary_model.joblib")

['best_salary_model.joblib']

### Predict for a new employee using the saved model

In [4]:
model = joblib.load("best_salary_model.joblib")   # full pipeline: imputing, scaling, encoding, SVM
new = pd.DataFrame([{
    "Age": 30, "Department": "Engineering", "Education": "Masters", "City_Tier": "Tier1",
    "Experience_Years": 6, "Performance_Rating": 3.8, "Weekly_Hours": 45, "Training_Hours": 30,
    "Certifications": 2, "Work_Life_Balance": 3.5, "Satisfaction": "High"}])
print("Predicted salary band:", model.predict(new)[0])

Predicted salary band: Medium
